# LG-FMEA-KG on XJTU-SY (Colab)

Colab driver for the paper *"Grounding Symbolic Failure-Mode Knowledge with
Physics: A Landau-Ginzburg Layer for Hybrid FMEA-Aware Fault Detection"*.

This notebook is intentionally thin: the scientific pipeline lives in the
`lg_fmea_kg` package in the repository, and this notebook only handles the
Colab-specific steps (clone, mount Drive, extract the `.rar` archives) and
then calls the package.

**Run order:** clone repo, mount Drive, extract dataset, discover bearings,
build dataset, run experiments, download results.


## 1. Clone the repository and install dependencies

In [ ]:
!git clone https://github.com/your-username/lg-fmea-kg.git
%cd lg-fmea-kg
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, "src")
print("Ready.")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Extract the .rar archives

The XJTU-SY mirror ships the dataset as multi-part `.rar` archives. Extract
them once into the Colab runtime's local disk (faster to read than Drive).
Set `RAR_DIR` to the folder on your Drive that holds the parts. The
extraction is lost when the session ends; re-run this cell in a new session,
or extract to a Drive path to keep it.


In [ ]:
import subprocess, time
from pathlib import Path

RAR_DIR     = Path("/content/drive/MyDrive/XJTU-SY_Bearing_Datasets/Data")
EXTRACT_DIR = Path("/content/xjtu_extracted")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

rar_parts = sorted(RAR_DIR.glob("*.rar"))
if not rar_parts:
    raise FileNotFoundError(f"No .rar files in {RAR_DIR}. Edit RAR_DIR.")
print(f"Found {len(rar_parts)} archive parts in {RAR_DIR}")

already = [p for p in EXTRACT_DIR.rglob("Bearing*") if p.is_dir()]
if len(already) >= 5:
    print(f"Already extracted: {len(already)} bearing folders. Skipping.")
else:
    subprocess.run(["apt-get", "install", "-y", "unrar"], check=True, capture_output=True)
    part01 = next((p for p in rar_parts if "part01" in p.name.lower()), rar_parts[0])
    print(f"Extracting {part01.name} (10-20 min)...")
    t0 = time.time()
    r = subprocess.run(["unrar", "x", "-y", str(part01), str(EXTRACT_DIR) + "/"],
                       capture_output=True, text=True)
    print(f"Done in {(time.time()-t0)/60:.1f} min, exit code {r.returncode}")
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError("rar extraction failed")

## 4. Discover bearings

In [ ]:
from lg_fmea_kg import discover_bearings
bearing_paths = discover_bearings(EXTRACT_DIR)

## 5. (Optional) Calibrate the LG thresholds

Defaults target roughly 50 to 70 percent HEALTHY snapshots. Leave
`LG_KWARGS` empty to use the defaults, or set thresholds explicitly.


In [ ]:
LG_KWARGS = {}
# Example: LG_KWARGS = {"eps_h": 0.3, "eps_c": 1.5}

## 6. Build the WC2 dataset

In [ ]:
import time
from lg_fmea_kg import build_dataset, FMEAGraph
t0 = time.time()
blocks = build_dataset(bearing_paths, subset="wc2", lg_kwargs=LG_KWARGS)
print(f"Built in {(time.time()-t0)/60:.1f} min")
kg = FMEAGraph(dim=8, seed=0)

## 7. Main 4-way ablation

In [ ]:
import numpy as np
from lg_fmea_kg import evaluate

results_main = {}
for mode in ["STAT", "STAT_KG", "STAT_LG", "STAT_LG_KG"]:
    rs = [evaluate(mode, kg, blocks, seed=s) for s in range(5)]
    results_main[mode] = {
        "precision": float(np.mean([r["precision"] for r in rs])),
        "recall":    float(np.mean([r["recall"]    for r in rs])),
        "f1":        float(np.mean([r["f1"]        for r in rs])),
        "f1_sd":     float(np.std([r["f1"]         for r in rs])),
        "n_feats":   int(rs[0]["n_feats"]),
    }
    m = results_main[mode]
    print(f"{mode:<14} feats={m['n_feats']:>3}  F1={m['f1']:.4f}±{m['f1_sd']:.3f}")

## 8. Scarcity stress-test, figures and download

In [ ]:
# The full scarcity sweep, figures and the results bundle are produced by the
# CLI runner. From a Colab cell you can call it directly:
!python run.py --data-dir /content/xjtu_extracted --experiment all

from google.colab import files
import shutil
shutil.make_archive("/content/xjtu_real_outputs", "zip", "results")
files.download("/content/xjtu_real_outputs.zip")